In [7]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier


## Loading the Analysis-Ready Dataset

This step loads the processed NHANES analysis-ready dataset from a Parquet file. This dataset was created after merging the selected NHANES cycles, harmonizing variables across cycles, constructing the diabetes outcome variable, applying the population criteria, selecting the required predictors, and renaming the columns to interpretable feature names.

After loading the dataset, the code prints the available columns to verify that the expected variables are present. It also displays the class distribution of the `diabetes` outcome variable, both as raw counts and as proportions. This is used to confirm the degree of class imbalance before applying preprocessing, train-test splitting, and model training.

In [8]:

df = pd.read_parquet("DATASETS/nhanes_diabetes_analysis_ready.parquet")
print(df.columns)
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))

Index(['participant_id', 'cycle', 'diabetes', 'age', 'sex', 'race_ethnicity',
       'education_level', 'income_poverty_ratio', 'bmi', 'waist_circumference',
       'mean_systolic_bp', 'mean_diastolic_bp', 'ever_smoked_100_cigarettes',
       'average_alcoholic_drinks_per_day', 'vigorous_work_activity',
       'moderate_work_activity', 'walk_or_bicycle_transport',
       'vigorous_recreational_activity', 'moderate_recreational_activity',
       'sleep_hours', 'energy_kcal', 'protein_g', 'carbohydrate_g',
       'total_sugar_g', 'fiber_g', 'total_fat_g', 'cholesterol_mg'],
      dtype='object')
diabetes
0    21092
1     4909
Name: count, dtype: Int64
diabetes
0    0.8112
1    0.1888
Name: proportion, dtype: Float64


## Splitting the Dataset into Training and Test Sets

This step separates the analysis-ready dataset into predictor variables (`X`) and the target variable (`y`). The participant identifier, survey cycle, and diabetes outcome are removed from `X`, because they should not be used as model predictors. The `diabetes` column is stored separately as the target variable.

The dataset is then split into a training set and a test set using an 80/20 split. The `stratify=y` argument ensures that the proportion of diabetic and non-diabetic participants remains approximately the same in both the training and test sets. This is important because the diabetes outcome is imbalanced.

The training set will be used for preprocessing, imbalance handling, and model training. The test set will remain untouched until final evaluation, so that model performance is measured on unseen data.

In [9]:
X = df.drop(columns=["participant_id", "cycle", "diabetes"])
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(20800, 24) (5201, 24)
diabetes
0    0.811202
1    0.188798
Name: proportion, dtype: Float64
diabetes
0    0.81119
1    0.18881
Name: proportion, dtype: Float64


## Handling Missing Values Using Training Data

This step handles missing values in the training and test sets. Separate copies of `X_train` and `X_test` are created so the original split remains unchanged.

Numerical variables are imputed using the median value calculated from the training set. The same training-set medians are then applied to the test set. Categorical variables are imputed using the most frequent value calculated from the training set, and the same values are applied to the test set.

This approach prevents data leakage, because no information from the test set is used to determine the imputation values. The test set remains unseen during the fitting of preprocessing steps and is only transformed using values learned from the training data.

In [10]:
numeric_features = [
    "age",
    "income_poverty_ratio",
    "bmi",
    "waist_circumference",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "average_alcoholic_drinks_per_day",
    "sleep_hours",
    "energy_kcal",
    "protein_g",
    "carbohydrate_g",
    "total_sugar_g",
    "fiber_g",
    "total_fat_g",
    "cholesterol_mg",
]

categorical_features = [
    "sex",
    "race_ethnicity",
    "education_level",
    "ever_smoked_100_cigarettes",
    "vigorous_work_activity",
    "moderate_work_activity",
    "walk_or_bicycle_transport",
    "vigorous_recreational_activity",
    "moderate_recreational_activity",
]

X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

numeric_medians = X_train_processed[numeric_features].median()

X_train_processed[numeric_features] = X_train_processed[numeric_features].fillna(numeric_medians)
X_test_processed[numeric_features] = X_test_processed[numeric_features].fillna(numeric_medians)

categorical_modes = X_train_processed[categorical_features].mode().iloc[0]

X_train_processed[categorical_features] = X_train_processed[categorical_features].fillna(categorical_modes)
X_test_processed[categorical_features] = X_test_processed[categorical_features].fillna(categorical_modes)

## Encoding Categorical Variables

This step converts categorical variables into numerical dummy variables using one-hot encoding. Although several NHANES categorical variables are stored as integer codes, these codes represent categories rather than continuous numerical values. One-hot encoding prevents the models from interpreting category codes as ordered numerical values.

The categorical variables are encoded separately for the training and test sets using `pd.get_dummies()`. The `drop_first=True` option removes one category from each encoded variable to reduce redundancy between dummy variables.

After encoding, the training and test sets are aligned to ensure that both datasets contain the same columns in the same order. This is necessary because some categories may appear in the training set but not in the test set, or vice versa. Missing dummy columns in the test set are filled with `0`, so the encoded feature matrices can be used consistently for model training and evaluation.

In [11]:
X_train_encoded = pd.get_dummies(
    X_train_processed,
    columns=categorical_features,
    drop_first=True
)

X_test_encoded = pd.get_dummies(
    X_test_processed,
    columns=categorical_features,
    drop_first=True
)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

## Scaling Numerical Features

This step standardizes the numerical predictor variables using `StandardScaler`. Standardization transforms each numerical feature so that it has a mean of approximately 0 and a standard deviation of 1.

The scaler is fitted only on the training set using `fit_transform()`. The same fitted scaler is then applied to the test set using `transform()`. This prevents data leakage, because information from the test set is not used to calculate the scaling parameters.

Scaling is especially important for models such as Logistic Regression, because these models are sensitive to differences in feature scale. Although tree-based models such as Decision Tree, Random Forest, and XGBoost do not strictly require scaling, the same scaled feature matrix is used to keep the preprocessing setup consistent across models.

In [12]:
scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train_scaled[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test_scaled[numeric_features])

## Defining the Final Preprocessed Feature Sets

This step assigns the fully preprocessed training and test sets to `X_train_final` and `X_test_final`. These datasets have already gone through missing-value imputation, categorical encoding, and numerical feature scaling.

The final feature sets will be used for both class-imbalance strategies. In Strategy A, SMOTE will be applied to `X_train_final` only. In Strategy B, the original `X_train_final` will be used with model-level class weighting. In both strategies, model performance will be evaluated on the same untouched `X_test_final`.

In [13]:
X_train_final = X_train_scaled
X_test_final = X_test_scaled

X_train_final.to_parquet("DATASETS/PREPROCESSED/X_train_final.parquet", index=False)

X_test_final.to_parquet("DATASETS/PREPROCESSED/X_test_final.parquet", index=False)

y_train.to_frame(name="diabetes").to_parquet("DATASETS/PREPROCESSED/y_train.parquet", index=False)

y_test.to_frame(name="diabetes").to_parquet("DATASETS/PREPROCESSED/y_test.parquet", index=False)